# Métricas Avanzadas
### MAPE (por qué engaña), sMAPE, FVA y el costo del error en pesos

`Fase 4 · Video 19` — serie **Forecasting de Ventas de la A a la Z**

Arranca la **Fase 4**. En el **Video 11** fijamos la brújula (WAPE, MASE, BIAS) y la usamos en toda la Fase 3. Aquí vamos
más allá de lo básico: por qué el **MAPE** engaña, qué aporta el **sMAPE**, si tu modelo **añade
valor** (FVA) y cuánto cuesta el error **en pesos**.

---
## 1. Librerías y carga de datos

In [ ]:
import warnings, sys
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.ticker import FuncFormatter, PercentFormatter
from statsmodels.tsa.holtwinters import ExponentialSmoothing
from pathlib import Path

plt.rcParams.update({
    'figure.facecolor': '#F8FAFC', 'axes.facecolor': '#F8FAFC',
    'axes.grid': True, 'grid.color': '#E2E8F0',
    'grid.linewidth': 0.8, 'font.size': 11,
})
BLUE, RED, GREEN, PURPLE, ORANGE = '#2563EB','#DC2626','#16A34A','#7C3AED','#EA580C'
unit_fmt = FuncFormatter(lambda x, _: f'{x/1e3:.0f}k')
print('✅ Librerías cargadas')

In [ ]:
def find_csv(filename='sales_history.csv', max_levels=4):
    base = Path().resolve()
    for _ in range(max_levels):
        candidate = base / 'output' / filename
        if candidate.exists():
            return candidate
        base = base.parent
    raise FileNotFoundError(f"No se encontró '{filename}'. Corre main.py primero.")

csv_path = find_csv()
sys.path.insert(0, str(csv_path.parents[1]))
from src.metrics import mae, wape, mape, smape, bias, mase, fva

df = pd.read_csv(csv_path, parse_dates=['date']).sort_values('date')
catalog = pd.read_csv(find_csv('sku_catalog.csv')).set_index('sku_id')

y = df.groupby(pd.Grouper(key='date', freq='W-MON'))['units_sold'].sum().iloc[1:-1]
H, m = 52, 52
train, test = y.iloc[:-H], y.iloc[-H:]

# Dos pronósticos reales sobre el total: seasonal naive y Holt-Winters
snaive = pd.Series(train.iloc[-m:].values[:H], index=test.index)
hw = ExponentialSmoothing(train, trend='add', seasonal='mul', seasonal_periods=m,
                          initialization_method='estimated').fit()
hw_f = pd.Series(hw.forecast(H).values, index=test.index)
print(f'✅ {len(y)} semanas | holdout {H}. Pronósticos base: Seasonal Naive y Holt-Winters.')

---
## 2. El zoológico de métricas

| Métrica | Fórmula (resumen) | Unidad | Fortaleza | Debilidad |
|---|---|---|---|---|
| **MAE** | media \|error\| | unidades | simple, robusta | no comparable entre series |
| **MAPE** | media \|error\|/\|real\| | % | intuitiva | **explota con ceros** |
| **WAPE** | Σ\|error\|/Σ\|real\| | % | robusta con ceros | — |
| **sMAPE** | media 2\|error\|/(\|real\|+\|pred\|) | % | acotada | interpretación rara |
| **BIAS** | Σ(pred−real)/Σ\|real\| | % con signo | detecta sesgo sistemático | no mide dispersión |
| **MASE** | MAE / MAE(naive) | ratio | comparable, con listón | necesita baseline |

Ninguna es "la mejor": WAPE, BIAS y MASE ya son la **brújula del Video 11**; aquí sumamos **MAPE**, **sMAPE** y las herramientas de decisión (FVA, costo en pesos). Se leen **juntas**. Veámoslas sobre nuestros dos pronósticos.

In [ ]:
def suite(name, f):
    return {'modelo': name, 'MAE': mae(test, f), 'WAPE': wape(test, f),
            'sMAPE': smape(test, f), 'BIAS': bias(test, f), 'MASE': mase(test, f, train, m)}

tabla = pd.DataFrame([suite('Seasonal Naive', snaive), suite('Holt-Winters', hw_f)]).set_index('modelo')
print(tabla.round(3).to_string())
print('\nEl Seasonal Naive tiene mejor WAPE/MASE. Fíjate también en el BIAS (signo): dice si el')
print('modelo tiende a quedarse corto (−) o a pasarse (+) — algo que el MAE nunca te dirá.')

---
## 3. El peligro del MAPE

El MAPE divide entre el valor real. Si hay **ceros** (demanda intermitente, Video 15), divide entre cero y
**explota** hacia infinito. Es la métrica más citada y la más peligrosa en retail. Lo mostramos en un SKU
intermitente.

In [ ]:
inter = catalog.index[catalog['demand_profile'] == 'Intermitente']
sku = df[df['sku_id'].isin(inter)].groupby('sku_id')['revenue'].sum().idxmax()
s = (df[df['sku_id'] == sku].groupby(pd.Grouper(key='date', freq='W-MON'))['units_sold']
     .sum().iloc[1:-1])
s_tr, s_te = s.iloc[:-H], s.iloc[-H:]
s_fc = pd.Series(s_tr.iloc[-m:].values[:H], index=s_te.index)   # seasonal naive del SKU

n_zeros = int((s_te == 0).sum())
print(f'SKU intermitente {sku}: {n_zeros} de {H} semanas con demanda CERO en el holdout\n')
print(f'  MAPE : {mape(s_te, s_fc):.2f}   ← {"∞ / sin sentido" if not np.isfinite(mape(s_te, s_fc)) else "inflado por los ceros"}')
print(f'  sMAPE: {smape(s_te, s_fc):.3f}  ← acotada, sobrevive')
print(f'  WAPE : {wape(s_te, s_fc):.3f}  ← robusta, la que usarías')
print(f'  MASE : {mase(s_te, s_fc, s_tr, m):.3f}')
print('\nRegla: en retail con ceros, NUNCA reportes MAPE. Usa WAPE (o MASE).')

---
## 4. BIAS: el error que el MAE esconde

Dos pronósticos pueden tener el **mismo** MAE y ser radicalmente distintos para el negocio: uno oscila
alrededor de lo real (insesgado), otro se pasa **siempre** (sesgado). El sesgo persistente es letal — se
acumula como sobre-inventario o quiebres. El MAE/WAPE no lo ven; el **BIAS** sí.

In [ ]:
rng = np.random.default_rng(0)
fc_unbiased = pd.Series(test.values * (1 + rng.normal(0, 0.10, H)), index=test.index)
fc_biased = pd.Series(test.values * 1.12, index=test.index)

fig, ax = plt.subplots(figsize=(14, 5))
ax.plot(test.index, test.values, color='black', linewidth=2.5, label='REAL')
ax.plot(test.index, fc_unbiased.values, color=GREEN, linewidth=1.5, label='A: insesgado')
ax.plot(test.index, fc_biased.values, color=RED, linewidth=1.5, label='B: sesgado (+12% siempre)')
ax.yaxis.set_major_formatter(unit_fmt)
ax.set_title('Mismo error absoluto, sesgo muy distinto', fontsize=13, fontweight='bold')
ax.legend(loc='upper left')
plt.tight_layout()
plt.show()

for name, f in [('A insesgado', fc_unbiased), ('B sesgado +12%', fc_biased)]:
    print(f'  {name:<16} WAPE {wape(test, f):.3f} | BIAS {bias(test, f):+.3f}')
print('\nWAPE parecido, pero B tiene BIAS ≈ +12%: cada semana sobra inventario. El MAE jamás')
print('te habría avisado. Monitorea el BIAS SIEMPRE: un sesgo persistente es una fuga de dinero.')

---
## 5. FVA: ¿tu modelo añade valor?

**Forecast Value Added** compara tu modelo contra un baseline trivial:

$$\text{FVA} = \frac{\text{error}_{\text{baseline}} - \text{error}_{\text{modelo}}}{\text{error}_{\text{baseline}}}$$

Positivo → el modelo **añade valor**. Cero o negativo → estás gastando cómputo y complejidad para **empeorar**
el baseline. Es la pregunta más honesta de todo el forecasting.

In [ ]:
fva_hw = fva(test, hw_f, snaive)          # Holt-Winters vs seasonal naive
# SARIMAX del Video 13 tuvo WAPE 4.9% vs seasonal naive 8.9%:
sarimax_wape = 0.049
fva_sarimax = (wape(test, snaive) - sarimax_wape) / wape(test, snaive)

fig, ax = plt.subplots(figsize=(9, 4.2))
vals = [fva_hw, fva_sarimax]
labels = ['Holt-Winters (V11)', 'SARIMAX (V12)']
ax.barh(labels, vals, color=[RED if v <= 0 else GREEN for v in vals])
ax.axvline(0, color='black', linewidth=1.2)
ax.xaxis.set_major_formatter(PercentFormatter(1.0))
ax.set_title('Forecast Value Added vs. Seasonal Naive', fontsize=13, fontweight='bold')
for i, v in enumerate(vals):
    ax.text(v, i, f' {v:+.0%}', va='center', fontweight='bold')
plt.tight_layout()
plt.show()

print(f'FVA Holt-Winters: {fva_hw:+.1%}  → {"añade valor" if fva_hw>0 else "❌ NO añade valor (peor que copiar el año pasado)"}')
print(f'FVA SARIMAX     : {fva_sarimax:+.1%}  → añade valor real')
print('\nLección brutal: un Holt-Winters "serio" puede tener FVA NEGATIVO. Sin medir FVA lo habrías')
print('desplegado creyendo que ayudabas. El baseline no es el rival a ignorar: es el rival a vencer.')

---
## 6. El costo en pesos del error

Las métricas cobran sentido cuando se traducen a dinero. Un **sesgo** persistente al alza infla el
inventario: mantienes stock de más y pagas su **costo de almacenamiento** (capital inmovilizado, bodega,
merma). Cuantifiquémoslo para nuestro total.

In [ ]:
holding_rate = 0.25            # costo anual de mantener inventario (% del valor)
avg_price = df['unit_price'].mean()
annual_units = test.sum()      # ~un año de demanda (holdout)

for b in [0.05, 0.12, 0.20]:
    exceso_unid = annual_units * b
    costo = exceso_unid * avg_price * holding_rate
    print(f'  Sesgo +{b:.0%}: sobras ~{exceso_unid:,.0f} unid/año → costo de almacenamiento ≈ ${costo:,.0f}/año')

# costo por punto de WAPE (aprox., vía stock de seguridad ~ error)
costo_punto = annual_units * 0.01 * avg_price * holding_rate
print(f'\nRegla de bolsillo: cada +1% de sesgo en el total ≈ ${costo_punto:,.0f}/año en inventario inmovilizado.')
print('Multiplícalo por cientos de SKUs: por eso el BIAS y el FVA no son métricas académicas, son P&L.')

---
## 7. Conclusiones y puente al Video 20

### Las reglas que te llevas

1. **Ninguna métrica basta sola.** Lee WAPE (magnitud), BIAS (dirección) y MASE/FVA (¿supera al baseline?).
2. **MAPE está prohibido con ceros.** En retail intermitente explota; usa **WAPE**.
3. **Vigila el BIAS.** Un sesgo persistente es invisible al MAE y carísimo en inventario.
4. **FVA es la prueba de fuego:** si tu modelo no le gana al baseline, no lo despliegues — aunque sea
   sofisticado (Holt-Winters aquí tuvo FVA negativo).
5. **Traduce a pesos.** El error no es un número: es capital inmovilizado o ventas perdidas.

### Lo que falta para confiar en el número

Hemos medido todo con **una sola partición** (los últimos 52 semanas). Pero, ¿y si ese año fue atípico?
Un solo hold-out puede engañarte. Necesitamos validar como manda el tiempo.

---

### Próximo video
**Video 20 — Validación cruzada temporal: walk-forward, expanding y sliding**
El error #1 que hunde proyectos en producción: validar una serie de tiempo como si fuera un problema
tabular. Los tres esquemas y cuál elegir.